# Module 10 — Vision Transformers (SOLUTIONS)

In [ ]:
import torch
import torch.nn as nn
import numpy as np

class RelativePositionBias(nn.Module):
    """Learnable relative position bias for Swin Transformer window attention."""
    def __init__(self, window_size=7):
        super().__init__()
        self.window_size = window_size
        Ws = window_size
        # Table has (2W-1) x (2W-1) entries for each head direction
        self.table = nn.Parameter(torch.zeros((2*Ws-1)**2))
        nn.init.trunc_normal_(self.table, std=0.02)

        # Pre-compute relative index matrix
        coords = torch.arange(Ws)
        grid_y, grid_x = torch.meshgrid(coords, coords, indexing='ij')
        coords_flat = torch.stack([grid_y.flatten(), grid_x.flatten()])  # (2, Ws*Ws)
        rel = coords_flat[:, :, None] - coords_flat[:, None, :]          # (2, N, N)
        rel[0] += Ws - 1
        rel[1] += Ws - 1
        rel[0] *= 2 * Ws - 1
        self.register_buffer('rel_idx', rel.sum(dim=0))  # (N, N)

    def forward(self):
        """Return (Ws*Ws, Ws*Ws) bias matrix."""
        return self.table[self.rel_idx.view(-1)].view(self.window_size**2, self.window_size**2)

# Test
rpb = RelativePositionBias(window_size=7)
bias = rpb()
print('Bias shape:', bias.shape)  # (49, 49)
print('Unique biases:', rpb.table.numel())  # (2*7-1)^2 = 169